🗺️👑 01 — When to Use Each: Who-Is-King Decision Guide

**Topic 12.04 | Async/Parallel Python — Part 4 of 4 (finale!)**

🌍 **Hook:** Interviewer: "1000 downloads — thread, process or async?"
You (confident 😎): "Lots of WAIT → async; CPU-crunch-style → processes;
IO with old sync-libs → threads." One question, three career-signals!

This notebook is section-12's MASTER-CHEAT:
- 📊 a one-look decision-tree (ASCII!)
- 🐣 a 6-scenario quiz (think first, then open the code!)
- 🐥 a small `recommend()` helper — a verdict from three facts
- 🐔 production-combos (AsyncIO + ThreadPool executor-shim, pros+cons)
- ⚠️ 7 deadly "wrong-tool" mistakes ~ + the golden recap!

## 📊 Master Decision-Tree (not memorized — UNDERSTOOD!)

```
                    kaam ka NATURE?
                ┌─────────┴──────────┐
             WAIT-zyada            CRUNCH-zyada
             (network,            (maths, image,
              files, DB)           resize, ML-transform)
                │                      │
       kitne waits baar-baar?      GIL ka shikar? (pure-Python tight loop)
       ┌────────┴─────────┐              │
   HAZAR+ parallel     thode (<100)   ┌──┴───┐
   (API storms,        + existing    HAan   C/Fortran under-hood
    chat sockets,      sync-libs     (numpy publish-karta GIL —
    crawlers)          (paths lib)    │      toh threads bhi ok!)
       │                  │            │
     ASYNC/AWAIT        THREADS    MULTIPROCESSING
     (ek waiter,       (counter-    (naya kitchen-hall
      hazaar tray!) 🦸   boys!) 🧵    + stove!) 🏭
```

🔍 **WHY does this tree work?**
- During WAIT the CPU is free → 1 waiter (async) is the cheapest; 100 waiters (threads) cost
  more; a fresh-kitchen (process) is not even a solution for waiting!
- During CRUNCH the GIL locks things: pure-Python threads don't work; **processes-only** is the path!
- numpy/pandas release the GIL (C-level) → a favor: threads are sometimes OK!

## 🐣 Level 0 — 6-Scenario Quiz (think for yourself first!)

Below, guess each scenario's verdict before running it, then run the cell and
see the verdict — plus WHY in two lines of reasoning!

In [1]:
def recommend(waits_parallel, cpu_crunch, sync_libs=False):
    """chhota decision-helper:
    waits_parallel = baar-baar wait? (bool)
    cpu_crunch     = pure-Python CPU loop? (bool)
    sync_libs      = purani NON-async libs use kar rahe? (bool)
    """
    if cpu_crunch:
        return ("🏭 PROCESSES", "GIL-tala pure-Python — naya-kitchen-hall (mp)!")
    if waits_parallel and not sync_libs:
        return ("🦸 ASYNC", "wait-storm + async-friendly libs = ek-waiter hazaar-tray!")
    if waits_parallel and sync_libs:
        return ("🧵 THREADS", "sync-libs ko async nahi ban sakte — counter-boys rescue!")
    return ("📦 PLAIN", "na wait na crunch — seedha loop (khatam-kahani!)")

scenarios = [
    ("1️⃣ 500 products ka REST-API scrape (httpx)", dict(waits_parallel=True, cpu_crunch=False)),
    ("2️⃣ 1000 images ka custom-Python pixel-filter", dict(waits_parallel=False, cpu_crunch=True)),
    ("3️⃣ 50 db-queries using OLD requests+sqlalchemy-sync", dict(waits_parallel=True, cpu_crunch=False, sync_libs=True)),
    ("4️⃣ numpy vector math 10M rows", dict(waits_parallel=False, cpu_crunch=True)),  # note: C-release → threads bhi chal jate, mp safest teach
    ("5️⃣ 5 audio files ka API-upload", dict(waits_parallel=True, cpu_crunch=False, sync_libs=True)),
    ("6️⃣ ek chhoti fib(30) ek-hi baar", dict(waits_parallel=False, cpu_crunch=False)),
]
for label, kw in scenarios:
    verdict, why = recommend(**kw)
    print(f"{label}\n    → {verdict} | {why}")
# Expected OUTPUT:
# 1️⃣ 500 products ka REST-API scrape (httpx)
#     → 🦸 ASYNC | wait-storm + async-friendly libs = ek-waiter hazaar-tray!
# 2️⃣ 1000 images ka custom-Python pixel-filter
#     → 🏭 PROCESSES | GIL-tala pure-Python — naya-kitchen-hall (mp)!
# 3️⃣ 50 db-queries using OLD requests+sqlalchemy-sync
#     → 🧵 THREADS | sync-libs ko async nahi ban sakte — counter-boys rescue!
# 4️⃣ numpy vector math 10M rows
#     → 🏭 PROCESSES | ... (safety: pure crunch = mp; numpy C-loops pe threads-se-bhi boost ho sakta — par simple-rule = mp!)
# 5️⃣ 5 audio files ka API-upload
#     → 🧵 THREADS | sync-libs ko async nahi ban sakte — counter-boys rescue!
# 6️⃣ ek chhoti fib(30) ek-hi baar
#     → 📦 PLAIN | na wait na crunch — seedha loop (khatam-kahani!)

1️⃣ 500 products ka REST-API scrape (httpx)
    → 🦸 ASYNC | wait-storm + async-friendly libs = ek-waiter hazaar-tray!
2️⃣ 1000 images ka custom-Python pixel-filter
    → 🏭 PROCESSES | GIL-tala pure-Python — naya-kitchen-hall (mp)!
3️⃣ 50 db-queries using OLD requests+sqlalchemy-sync
    → 🧵 THREADS | sync-libs ko async nahi ban sakte — counter-boys rescue!
4️⃣ numpy vector math 10M rows
    → 🏭 PROCESSES | GIL-tala pure-Python — naya-kitchen-hall (mp)!
5️⃣ 5 audio files ka API-upload
    → 🧵 THREADS | sync-libs ko async nahi ban sakte — counter-boys rescue!
6️⃣ ek chhoti fib(30) ek-hi baar
    → 📦 PLAIN | na wait na crunch — seedha loop (khatam-kahani!)


## 🐥 Level 1 — Scorecard: All Three Tools, 6 Metrics 🏆

| Metric | 🦸 async | 🧵 threads | 🏭 processes |
|---|---|---|---|
| IO-wait storms | 🥇🥇 | 🥈 | 🥉 |
| CPU-crunch (pure-py) | ❌ | ❌(GIL) | 🥇🥇 |
| scale to thousands of connections | 🥇🥇 | OS-limit ~ | ❌ heavy |
| with sync-libs | ❌ needs-async | 🥇 | 😐 |
| debugging ease | 🥉 (slick) | 🥈 | 🥇 (isolated) |
| code-complexity | 😅 async/await everywhere | 😊 easy-start | 😐 IPC-hassle |

In [2]:
# Live-mini: teeno "cost" ek-tera-feel — dispatch table!
TOOLBOX = {
    "async":     {"wait_storm": 10, "cpu_crunch": 2, "hazar_conn": 10, "sync_libs": 2},
    "threads":   {"wait_storm": 7,  "cpu_crunch": 1, "hazar_conn": 4,  "sync_libs": 9},
    "processes": {"wait_storm": 4,  "cpu_crunch": 10, "hazar_conn": 1, "sync_libs": 5},
}
def score(tool, scenario_weights):
    return sum(TOOLBOX[tool][k] * w for k, w in scenario_weights.items())

challenges = {
    "API-storm": {"wait_storm": 3, "hazar_conn": 2},
    "CPU-crunch": {"cpu_crunch": 5},
    "sync-libs-io": {"wait_storm": 2, "sync_libs": 3},
}
for sc, w in challenges.items():
    board = sorted(((score(t, w), t) for t in TOOLBOX), reverse=True)
    print(f"{sc:12s} → " + "  ".join(f"{t}={s}" for s, t in board),
          f"   🏅 {board[0][1].upper()}")
# Expected OUTPUT:
# API-storm    → async=50  threads=29  processes=14   🏅 ASYNC   (10*3+10*2!)
# CPU-crunch   → processes=50  async=10  threads=5   🏅 PROCESSES
# sync-libs-io → threads=41  async=26  processes=23   🏅 THREADS

API-storm    → async=50  threads=29  processes=14    🏅 ASYNC
CPU-crunch   → processes=50  async=10  threads=5    🏅 PROCESSES
sync-libs-io → threads=41  async=26  processes=23    🏅 THREADS


## 🐔 Level 2 — Production Combos (a mix of real apps!) 🍱

Real apps are never single-tool pure — they're a COMBO platter:
```python
# 🔗 1. A BLOCKING-lib inside AsyncIO? Use a ThreadPool shim:
loop = asyncio.get_running_loop()
result = await loop.run_in_executor(None, blocking_func, arg)
    # run sync-libs inside the async event-loop without choking!

# 🔗 2. CPU-crunch in the middle of async tasks? Use a ProcessPool executor:
result = await loop.run_in_executor(process_pool, heavy_calc, data)
    # the waiter's hand stays FREE while a stove runs in the kitchen-hall!

# 🔗 3. OCR pipeline: async(download) → thread(PIL) → process(ocr-CPU)
#    each stage uses its own best tool — that is production JUGAAD! 😎
```

---
## ⚠️ 7 Deadly "Wrong-Tool" Mistakes
| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | CPU-crunch with threads | processes (verify-GIL-skip!) |
| 2 | thinking async with 1000-simultaneous-sync libs | threads or async-refactor (httpx!) |
| 3 | treating mp as a "speed-God" everywhere | mp is heavy: only for TRUE crunch |
| 4 | sync `requests.get` inside fastapi | `httpx.AsyncClient` or run_in_executor |
| 5 | `time.sleep` in async-code | `asyncio.sleep` (loop-air!) |
| 6 | lambda in a process-pool (pickle-error!) | top-level funcs (`if __name__==...`) |
| 7 | skipping the scenario-quiz and memorizing | build your own guide (print this!) |

## 📌 Recap — Section-12's Golden Frame
- Big WAIT-Storm → **ASYNC** 🦸 (one waiter, Semaphore + wait_for shields!)
- Few IO sync-libs → **THREADS** 🧵 (counter-boys, the GIL doesn't interfere!)
- A CRUNCH-life → **PROCESSES** 🏭 (a new hall, `__main__` guard!)
- combos = the production-platter 🍱 · PEP/GIL keeps SHIFTING all your life (this guide stays alive!)

**Section 12 = COMPLETE!** 🎓 Next section: OOPS's five doors (13_OOPS_Concepts) 🚪